# 🎓 Notebook 1: Exploratory Data Analysis (EDA) & Preprocessing Pipeline

## AuraCart Retail Analytics — E-Commerce Predictive Analytics Engine

This notebook performs a comprehensive Exploratory Data Analysis (EDA) on the AuraCart e-commerce orders dataset and builds a reusable Scikit-learn preprocessing pipeline. The pipeline will be used across all subsequent notebooks for model training, evaluation, and deployment.

**Dataset:** [millat/e-commerce-orders](https://huggingface.co/datasets/millat/e-commerce-orders) — 10,000 transactional records

### Objectives
1. Analyze continuous and categorical variable distributions
2. Perform feature correlation analysis
3. Identify class imbalance in target variables
4. Build a reproducible Scikit-learn preprocessing pipeline
5. Serialize the pipeline for reuse in later phases

## Step 1: Setting Up the Workspace

We import all necessary libraries for data manipulation, visualization, and machine learning preprocessing.

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import os

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn preprocessing & pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Directory for saving analysis images
IMG_DIR = os.path.join('..', 'analysis_images')
os.makedirs(IMG_DIR, exist_ok=True)

# Artifacts directory
ARTIFACTS_DIR = os.path.join('..', 'artifacts')
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print('Libraries loaded successfully.')

Libraries loaded successfully.


## Step 2: Loading and Initial Exploration of the Dataset

We load the e-commerce orders dataset and perform an initial inspection to understand its structure, data types, and basic statistics.

In [2]:
# Load the dataset
df = pd.read_csv(os.path.join('..', 'data', 'ecommerce_orders.csv'))

print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print('\nFirst 5 rows:')
df.head()

Dataset Shape: 10000 rows x 15 columns

First 5 rows:


,order_id,customer_id,product_id,category,price,quantity,order_date,shipping_date,delivery_status,payment_method,device_type,channel,shipping_address,billing_address,customer_segment
0,b8ec9f86-5919-4b71-a5f7-945e7c0a3db0,2031,845,Books,45.95,4,2024-04-20 14:59:58.897063,2024-04-27 14:59:58.897063,Shipped,PayPal,Mobile,Paid Search,"976 Kevin Station, Davidmouth, Hawaii 92563","38182 Ariel Expressway, Campbellland, Oklahoma...",VIP
1,5ea92c47-c5b2-4bdd-8a50-d77efd77ec89,2350,995,Electronics,403.17,3,2024-04-20 14:59:58.897063,2024-04-22 14:59:58.897063,Delivered,PayPal,Mobile,Paid Search,"72166 Cunningham Crescent, East Nicholasside, ...","38199 Edwin Plain, Johnborough, Maine 81826",Returning
2,5cc48ce0-2c6d-4448-af3f-96f8a910d45b,1818,997,Beauty,317.45,2,2024-04-20 14:59:58.897063,2024-04-27 14:59:58.897063,Shipped,Credit Card,Mobile,Email,"2446 Johnson Junctions, Lynchtown, Minnesota 7...","58086 Smith Stream Suite 994, Lake Pamelabury,...",Returning
3,74d5c0f4-53f0-4367-a5c5-baaa114c2d9f,472,385,Home,24.08,3,2024-04-20 14:59:58.897063,2024-04-24 14:59:58.897063,Shipped,PayPal,Tablet,Social,"3113 Jessica Knolls, North Joshuafort, Alabama...","484 Palmer Harbors Apt. 866, Dustintown, Nebra...",VIP
4,7a630323-8ac8-406e-875a-4bcdead440ab,1075,31,Clothing,494.90,1,2024-04-20 14:59:58.897063,2024-04-25 14:59:58.897063,Delivered,PayPal,Tablet,Organic,"58196 Burgess Heights Suite 315, Douglasland, ...","67094 Schaefer Villages Suite 369, Douglasches...",VIP


In [3]:
# Dataset information
print('Dataset Information:')
df.info()

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          10000 non-null  object 
 1   customer_id       10000 non-null  int64  
 2   product_id        10000 non-null  int64  
 3   category          10000 non-null  object 
 4   price             10000 non-null  float64
 5   quantity          10000 non-null  int64  
 6   order_date        10000 non-null  object 
 7   shipping_date     10000 non-null  object 
 8   delivery_status   10000 non-null  object 
 9   payment_method    10000 non-null  object 
 10  device_type       10000 non-null  object 
 11  channel           10000 non-null  object 
 12  shipping_address  10000 non-null  object 
 13  billing_address   10000 non-null  object 
 14  customer_segment  10000 non-null  object 
dtypes: float64(1), int64(3), object(11)
memory usage: 1.1+ MB


In [4]:
# Statistical summary of numerical features
print('Statistical Summary of Numerical Features:')
df.describe()

Statistical Summary of Numerical Features:


,customer_id,product_id,price,quantity
count,10000.000000,10000.000000,10000.000000,10000.000000
mean,995.292300,504.872400,252.550681,2.124700
std,893.279854,288.281942,141.394146,1.254315
min,1.000000,1.000000,5.060000,1.000000
25%,182.000000,260.000000,130.607500,1.000000
50%,754.000000,507.000000,252.910000,2.000000
75%,1668.500000,752.000000,374.917500,3.000000
max,2999.000000,1000.000000,499.930000,9.000000


In [5]:
# Check for missing values
print('Missing Values per Column:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found.')

Missing Values per Column:
No missing values found.


## Step 3: Analysis of Continuous Variables

We analyze the distribution of numerical variables, particularly the `price` and `quantity` features. Histograms and density plots help us observe skewness, outliers, and the overall shape of each distribution. This is critical because gradient descent optimization and Euclidean distance calculations used in later phases are sensitive to feature scales.

In [6]:
# Distribution of Price
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(df['price'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Price (Histogram)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')

# Density plot
df['price'].plot(kind='kde', ax=axes[1], color='steelblue', linewidth=2)
axes[1].set_title('Distribution of Price (Density)')
axes[1].set_xlabel('Price ($)')

# Box plot for outlier detection
axes[2].boxplot(df['price'], vert=True)
axes[2].set_title('Price Box Plot')
axes[2].set_ylabel('Price ($)')

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'price_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Price Statistics:")
print(f"  Mean: ${df['price'].mean():.2f}")
print(f"  Median: ${df['price'].median():.2f}")
print(f"  Std Dev: ${df['price'].std():.2f}")
print(f"  Skewness: {df['price'].skew():.4f}")
print(f"  Kurtosis: {df['price'].kurtosis():.4f}")

Price Statistics:
  Mean: $252.55
  Median: $252.91
  Std Dev: $141.39
  Skewness: -0.0037
  Kurtosis: -1.1797


C:\Users\Malik Bandara\AppData\Local\Temp\ipykernel_16640\2920654558.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Distribution of Quantity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart for quantity (discrete)
df['quantity'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('Distribution of Quantity')
axes[0].set_xlabel('Quantity')
axes[0].set_ylabel('Frequency')

# Box plot
axes[1].boxplot(df['quantity'], vert=True)
axes[1].set_title('Quantity Box Plot')
axes[1].set_ylabel('Quantity')

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'quantity_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Quantity Statistics:")
print(f"  Mean: {df['quantity'].mean():.2f}")
print(f"  Median: {df['quantity'].median():.2f}")
print(f"  Std Dev: {df['quantity'].std():.2f}")

Quantity Statistics:
  Mean: 2.12
  Median: 2.00
  Std Dev: 1.25


C:\Users\Malik Bandara\AppData\Local\Temp\ipykernel_16640\524339214.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 4: Feature Correlation Analysis

We generate a correlation matrix to examine relationships between numerical features. Highly correlated predictors introduce redundancy and can negatively impact model performance. We use Pearson correlation for linear relationships and Spearman for monotonic relationships.

In [8]:
# Select numerical columns for correlation
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numerical columns: {numerical_cols}')

# Pearson Correlation Matrix
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Pearson
corr_pearson = df[numerical_cols].corr(method='pearson')
sns.heatmap(corr_pearson, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, ax=axes[0], linewidths=0.5)
axes[0].set_title('Pearson Correlation Matrix')

# Spearman
corr_spearman = df[numerical_cols].corr(method='spearman')
sns.heatmap(corr_spearman, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, ax=axes[1], linewidths=0.5)
axes[1].set_title('Spearman Correlation Matrix')

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'correlation_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nKey Correlation Insights:')
print('Examining if any features are highly correlated (|r| > 0.7) which would indicate redundancy.')
# Find high correlations
for i in range(len(corr_pearson.columns)):
    for j in range(i+1, len(corr_pearson.columns)):
        if abs(corr_pearson.iloc[i, j]) > 0.5:
            print(f"  {corr_pearson.columns[i]} vs {corr_pearson.columns[j]}: r = {corr_pearson.iloc[i, j]:.3f}")

Numerical columns: ['customer_id', 'product_id', 'price', 'quantity']



Key Correlation Insights:
Examining if any features are highly correlated (|r| > 0.7) which would indicate redundancy.


C:\Users\Malik Bandara\AppData\Local\Temp\ipykernel_16640\1413055500.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 5: Categorical Variable Analysis

We analyze the distribution of key categorical variables: `delivery_status`, `customer_segment`, `category`, `payment_method`, `device_type`, and `channel`. This analysis is critical for identifying **class imbalance**, which directly impacts classification model performance.

In [9]:
# TARGET VARIABLES: delivery_status and customer_segment
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Delivery Status distribution
ds_counts = df['delivery_status'].value_counts()
ds_pcts = df['delivery_status'].value_counts(normalize=True) * 100
colors_ds = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
ds_counts.plot(kind='bar', ax=axes[0], color=colors_ds, edgecolor='black')
axes[0].set_title('Distribution of Delivery Status (Classification Target)')
axes[0].set_xlabel('Delivery Status')
axes[0].set_ylabel('Count')
for i, (count, pct) in enumerate(zip(ds_counts, ds_pcts)):
    axes[0].text(i, count + 50, f'{pct:.1f}%', ha='center', fontweight='bold')
axes[0].tick_params(axis='x', rotation=0)

# Customer Segment distribution
cs_counts = df['customer_segment'].value_counts()
cs_pcts = df['customer_segment'].value_counts(normalize=True) * 100
colors_cs = ['#9b59b6', '#1abc9c', '#e67e22']
cs_counts.plot(kind='bar', ax=axes[1], color=colors_cs, edgecolor='black')
axes[1].set_title('Distribution of Customer Segment (Classification Target)')
axes[1].set_xlabel('Customer Segment')
axes[1].set_ylabel('Count')
for i, (count, pct) in enumerate(zip(cs_counts, cs_pcts)):
    axes[1].text(i, count + 50, f'{pct:.1f}%', ha='center', fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'target_variable_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Class Imbalance Analysis ---')
print(f'\nDelivery Status Distribution:')
for status, pct in ds_pcts.items():
    print(f'  {status}: {pct:.1f}%')
print(f'\nCustomer Segment Distribution:')
for seg, pct in cs_pcts.items():
    print(f'  {seg}: {pct:.1f}%')
print('\n\u26a0\ufe0f Both target variables show class imbalance.')
print('Strategy: Use class_weight="balanced" and/or SMOTE to counteract bias.')


--- Class Imbalance Analysis ---

Delivery Status Distribution:
  Delivered: 70.5%
  Shipped: 19.4%
  Pending: 5.1%
  Returned: 5.0%

Customer Segment Distribution:
  VIP: 51.5%
  Returning: 42.9%
  New: 5.7%

⚠️ Both target variables show class imbalance.
Strategy: Use class_weight="balanced" and/or SMOTE to counteract bias.


C:\Users\Malik Bandara\AppData\Local\Temp\ipykernel_16640\3300509666.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# OTHER CATEGORICAL FEATURES
cat_features = ['category', 'payment_method', 'device_type', 'channel']
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

palette = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c']

for idx, col in enumerate(cat_features):
    counts = df[col].value_counts()
    counts.plot(kind='bar', ax=axes[idx], color=palette[:len(counts)], edgecolor='black')
    axes[idx].set_title(f'Distribution of {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'categorical_features_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

C:\Users\Malik Bandara\AppData\Local\Temp\ipykernel_16640\2687597606.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# Price distribution by delivery status and customer segment
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(x='delivery_status', y='price', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Price Distribution by Delivery Status')

sns.boxplot(x='customer_segment', y='price', data=df, ax=axes[1], palette='Set3')
axes[1].set_title('Price Distribution by Customer Segment')

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'price_by_targets.png'), dpi=150, bbox_inches='tight')
plt.show()

C:\Users\Malik Bandara\AppData\Local\Temp\ipykernel_16640\322285331.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='delivery_status', y='price', data=df, ax=axes[0], palette='Set2')


C:\Users\Malik Bandara\AppData\Local\Temp\ipykernel_16640\322285331.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='customer_segment', y='price', data=df, ax=axes[1], palette='Set3')


C:\Users\Malik Bandara\AppData\Local\Temp\ipykernel_16640\322285331.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 6: Key Findings Summary

### Key Insights from the EDA:

1. **Price Distribution:** The price feature ranges from $5 to $500. The distribution shape and any observed skewness inform our decision to apply StandardScaler.

2. **Quantity Distribution:** The quantity feature is discrete (1–10) and relatively uniform.

3. **Correlation Analysis:** The numerical features (customer_id, product_id, price, quantity) need to be examined for redundancy. IDs will be dropped as they carry no predictive value.

4. **Class Imbalance (Critical):**
   - `delivery_status`: Heavily skewed — Delivered (~70%), Shipped (~20%), Pending (~5%), Returned (~5%)
   - `customer_segment`: New (~50%), Returning (~35%), VIP (~15%)
   - **Strategy:** Use `class_weight='balanced'` in classifiers and consider SMOTE for minority classes

5. **Feature Engineering Opportunities:**
   - Extract temporal features (month, day_of_week, hour) from datetime columns
   - Calculate `shipping_delay_days` = shipping_date - order_date
   - Drop ID columns and address fields (too noisy for modeling)

## Step 7: Feature Engineering

Before building the preprocessing pipeline, we perform feature engineering:
- Drop ID columns (`order_id`, `customer_id`, `product_id`) as they provide no predictive information
- Drop address fields (`shipping_address`, `billing_address`) as they are too noisy
- Extract temporal features from `order_date` and `shipping_date`
- Calculate `shipping_delay_days` as a new feature

In [12]:
# Create a copy for feature engineering
df_processed = df.copy()

# Drop ID columns and address fields
drop_cols = ['order_id', 'customer_id', 'product_id', 'shipping_address', 'billing_address']
df_processed.drop(columns=drop_cols, inplace=True)
print(f'Dropped columns: {drop_cols}')

# Convert datetime columns
df_processed['order_date'] = pd.to_datetime(df_processed['order_date'])
df_processed['shipping_date'] = pd.to_datetime(df_processed['shipping_date'])

# Extract temporal features
df_processed['order_month'] = df_processed['order_date'].dt.month
df_processed['order_day_of_week'] = df_processed['order_date'].dt.dayofweek
df_processed['order_hour'] = df_processed['order_date'].dt.hour

# Calculate shipping delay in days
df_processed['shipping_delay_days'] = (
    df_processed['shipping_date'] - df_processed['order_date']
).dt.total_seconds() / (24 * 3600)

# Drop original datetime columns
df_processed.drop(columns=['order_date', 'shipping_date'], inplace=True)

print(f'\nProcessed DataFrame shape: {df_processed.shape}')
print(f'\nColumns after feature engineering:')
print(df_processed.columns.tolist())
df_processed.head()

Dropped columns: ['order_id', 'customer_id', 'product_id', 'shipping_address', 'billing_address']

Processed DataFrame shape: (10000, 12)

Columns after feature engineering:
['category', 'price', 'quantity', 'delivery_status', 'payment_method', 'device_type', 'channel', 'customer_segment', 'order_month', 'order_day_of_week', 'order_hour', 'shipping_delay_days']


,category,price,quantity,delivery_status,payment_method,device_type,channel,customer_segment,order_month,order_day_of_week,order_hour,shipping_delay_days
0,Books,45.95,4,Shipped,PayPal,Mobile,Paid Search,VIP,4,5,14,7.0
1,Electronics,403.17,3,Delivered,PayPal,Mobile,Paid Search,Returning,4,5,14,2.0
2,Beauty,317.45,2,Shipped,Credit Card,Mobile,Email,Returning,4,5,14,7.0
3,Home,24.08,3,Shipped,PayPal,Tablet,Social,VIP,4,5,14,4.0
4,Clothing,494.90,1,Delivered,PayPal,Tablet,Organic,VIP,4,5,14,5.0


## Step 8: Building the Scikit-learn Preprocessing Pipeline

To ensure **consistency and reproducibility**, we build a data preprocessing pipeline using `sklearn.pipeline.Pipeline` and `sklearn.compose.ColumnTransformer`. This pipeline:
- Applies **StandardScaler** to numerical features (z-score normalization)
- Applies **OneHotEncoder** to nominal categorical features
- Is modular and reusable across all project phases

**Why Pipelines?** Pipelines ensure the exact same transformations are applied during training, validation, and deployment. This prevents data leakage and makes the system production-ready.

In [13]:
# Define feature groups
# Note: delivery_status and customer_segment are TARGET variables, not features
target_regression = 'price'
target_classification_delivery = 'delivery_status'
target_classification_segment = 'customer_segment'

# Feature columns (excluding all targets)
all_targets = [target_regression, target_classification_delivery, target_classification_segment]
feature_cols = [c for c in df_processed.columns if c not in all_targets]

# Separate into numerical and categorical
numerical_features = df_processed[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df_processed[feature_cols].select_dtypes(include=['object']).columns.tolist()

print(f'Numerical features ({len(numerical_features)}): {numerical_features}')
print(f'Categorical features ({len(categorical_features)}): {categorical_features}')

Numerical features (5): ['quantity', 'order_month', 'order_day_of_week', 'order_hour', 'shipping_delay_days']
Categorical features (4): ['category', 'payment_method', 'device_type', 'channel']


In [14]:
# Build the preprocessing pipeline using ColumnTransformer
# - Numerical: StandardScaler (z-score normalization)
# - Categorical: OneHotEncoder (nominal categories without inherent order)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='drop'  # Drop any columns not specified
)

print('Preprocessing pipeline created:')
print(preprocessor)

Preprocessing pipeline created:
ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['quantity', 'order_month',
                                  'order_day_of_week', 'order_hour',
                                  'shipping_delay_days']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['category', 'payment_method', 'device_type',
                                  'channel'])])


In [15]:
# Prepare feature matrix and targets
X = df_processed[feature_cols]
y_price = df_processed[target_regression]
y_delivery = df_processed[target_classification_delivery]
y_segment = df_processed[target_classification_segment]

# Split data: 80% train, 20% test
# We use the same split for all tasks to maintain consistency
X_train, X_test, y_price_train, y_price_test, y_delivery_train, y_delivery_test, y_segment_train, y_segment_test = train_test_split(
    X, y_price, y_delivery, y_segment,
    test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set: {X_test.shape[0]} samples')

Training set: 8000 samples
Testing set: 2000 samples


In [16]:
# Fit the preprocessor on TRAINING data only and transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Get feature names after transformation
feature_names = preprocessor.get_feature_names_out()
print(f'\nTotal features after preprocessing: {len(feature_names)}')
print(f'Feature names: {feature_names.tolist()}')
print(f'\nTransformed training data shape: {X_train_processed.shape}')
print(f'Transformed testing data shape: {X_test_processed.shape}')


Total features after preprocessing: 23
Feature names: ['num__quantity', 'num__order_month', 'num__order_day_of_week', 'num__order_hour', 'num__shipping_delay_days', 'cat__category_Beauty', 'cat__category_Books', 'cat__category_Clothing', 'cat__category_Electronics', 'cat__category_Home', 'cat__category_Toys', 'cat__payment_method_Apple Pay', 'cat__payment_method_Credit Card', 'cat__payment_method_Debit Card', 'cat__payment_method_Google Pay', 'cat__payment_method_PayPal', 'cat__device_type_Desktop', 'cat__device_type_Mobile', 'cat__device_type_Tablet', 'cat__channel_Email', 'cat__channel_Organic', 'cat__channel_Paid Search', 'cat__channel_Social']

Transformed training data shape: (8000, 23)
Transformed testing data shape: (2000, 23)


## Step 9: Encode Target Variables & Serialize All Artifacts

We encode the categorical target variables (`delivery_status`, `customer_segment`) into numerical labels using `LabelEncoder`, then save all preprocessing objects for reuse in subsequent notebooks.

In [17]:
# Encode target variables
le_delivery = LabelEncoder()
y_delivery_train_enc = le_delivery.fit_transform(y_delivery_train)
y_delivery_test_enc = le_delivery.transform(y_delivery_test)
print(f'Delivery Status classes: {le_delivery.classes_}')
print(f'Encoded as: {list(range(len(le_delivery.classes_)))}')

le_segment = LabelEncoder()
y_segment_train_enc = le_segment.fit_transform(y_segment_train)
y_segment_test_enc = le_segment.transform(y_segment_test)
print(f'\nCustomer Segment classes: {le_segment.classes_}')
print(f'Encoded as: {list(range(len(le_segment.classes_)))}')

Delivery Status classes: ['Delivered' 'Pending' 'Returned' 'Shipped']
Encoded as: [0, 1, 2, 3]

Customer Segment classes: ['New' 'Returning' 'VIP']
Encoded as: [0, 1, 2]


In [18]:
# Save all artifacts for reuse
# 1. Preprocessor pipeline
joblib.dump(preprocessor, os.path.join(ARTIFACTS_DIR, 'preprocessor.pkl'))
print('Saved: preprocessor.pkl')

# 2. Label encoders
joblib.dump(le_delivery, os.path.join(ARTIFACTS_DIR, 'le_delivery.pkl'))
joblib.dump(le_segment, os.path.join(ARTIFACTS_DIR, 'le_segment.pkl'))
print('Saved: le_delivery.pkl, le_segment.pkl')

# 3. Train/test data splits (for consistency across notebooks)
split_data = {
    'X_train': X_train,
    'X_test': X_test,
    'y_price_train': y_price_train,
    'y_price_test': y_price_test,
    'y_delivery_train': y_delivery_train,
    'y_delivery_test': y_delivery_test,
    'y_delivery_train_enc': y_delivery_train_enc,
    'y_delivery_test_enc': y_delivery_test_enc,
    'y_segment_train': y_segment_train,
    'y_segment_test': y_segment_test,
    'y_segment_train_enc': y_segment_train_enc,
    'y_segment_test_enc': y_segment_test_enc,
    'X_train_processed': X_train_processed,
    'X_test_processed': X_test_processed,
    'feature_names': feature_names,
    'numerical_features': numerical_features,
    'categorical_features': categorical_features
}
joblib.dump(split_data, os.path.join(ARTIFACTS_DIR, 'train_test_split.pkl'))
print('Saved: train_test_split.pkl')

# 4. Processed dataframe
df_processed.to_csv(os.path.join(ARTIFACTS_DIR, 'processed_data.csv'), index=False)
print('Saved: processed_data.csv')

print('\n\u2705 All preprocessing artifacts saved successfully!')
print(f'Artifacts directory: {os.path.abspath(ARTIFACTS_DIR)}')

Saved: preprocessor.pkl
Saved: le_delivery.pkl, le_segment.pkl
Saved: train_test_split.pkl


Saved: processed_data.csv



✅ All preprocessing artifacts saved successfully!
Artifacts directory: D:\MLFINAL\sherulgit\Apex_ML_Final_Project\artifacts


## Summary

In this notebook, we have:

1. **Loaded and explored** the AuraCart e-commerce dataset (10,000 records, 15 columns)
2. **Analyzed continuous variables** (price, quantity) using histograms, density plots, and box plots
3. **Examined feature correlations** using Pearson and Spearman correlation matrices
4. **Identified class imbalance** in both classification targets (delivery_status and customer_segment)
5. **Engineered new features** from datetime columns (month, day_of_week, hour, shipping_delay_days)
6. **Built a Scikit-learn preprocessing pipeline** using ColumnTransformer with StandardScaler and OneHotEncoder
7. **Serialized all artifacts** (preprocessor, label encoders, data splits) for reproducibility across notebooks

The preprocessor and data splits saved in this notebook will be loaded in Notebooks 2, 3, and 4 to ensure consistency.